In [ ]:
# ======================================
# ファインチューニングの実装
# ======================================
!pip install transformers torch torchvision
import transformers
from transformers import Trainer, TrainingArguments, ViTImageProcessor, ViTForImageClassification
from torch.utils.data import Dataset
import torch
from PIL import Image
import os
from sklearn.model.selection import accuracy_score, precision_recall_fscore_support

# ======================================
# カスタムデータセットクラスの定義
# ======================================

class CustomImageDataset(Dataset):
    """
    カスタムデータセットクラス
    
    PyTorchのDatasetクラスを継承し、画像データとラベルを扱うためのカスタムデータセットクラスを定義します。
    __len__メソッドではデータセットのサイズを返し、__getitem__メソッドでは指定されたインデックスに対応する画像とラベルを返します。
    """

    def __init__(self, image_paths, labels, processor):
        """
        データセットの初期化
        
        Args:
            image_paths (list): 画像ファイルのパスのリスト
            labels (list): 画像に対応するラベルのリスト
            processor (ViTImageProcessor): 画像を前処理するためのプロセッサ
        """
        self.image_paths = image_paths
        self.labels = labels
        self.processor = processor

    def __len__(self):
        """データセットのサイズを返す"""
        return len(self.image_paths)

    def __getitem__(self, idx):
        """
            指定されたインデックスに対応する画像とラベルを返す
            returns:
                dict: {'pixel_values': tensor, 'label': int}
        """
        # 画像を読み込み、RGBに変換
        image = Image.open(self.image_paths[idx]).convert("RGB")
        # 前処理を適用
        inputs = self.processor(image, return_tensors="pt")

        # バッチ次元を削除(Trainerが後で追加する)
        # (1, 3, 224, 224) -> (3, 224, 224)
        pixel_values = inputs['pixel_values'].squeeze(0)

        return {
            'pixel_values': pixel_values,
            'label': torch.tensor(self.labels[idx], dtype=torch.long)
            }

In [ ]:
# ======================================
# データの準備
# ======================================

# 実際のプロジェクトでは、以下のようにファイルパスとラベルを準備
# ここでは例として構造を示す

# データディレクトリの構造:
# data/
#   train/
#     dog/
#       img1.jpg
#       img2.jpg
#       …
#     cat/
#       img1.jpg
#       img2.jpg
#       …
#   val/
#     dog/
#       …
#     cat/
#       …

def load_dataset_paths(data_dir, class_names):
    """
    データセットのパスとラベルを読み込む
    
    Args:
        data_dir (str): データディレクトリのパス
        class_names (list): クラス名のリスト
    
    Returns:
        image_paths (list): 画像ファイルのパスのリスト
        labels (list): 画像に対応するラベルのリスト
    """
    image_paths = []
    labels = []

    for label_idx, class_name in enumerate(class_names):
        class_dir = os.path.join(data_dir, class_name)

        if os.path.exists(class_dir):
            for filename in os.listdir(class_dir):
                if filename.lower().endswith(('.png', '.jpg', '.jpeg')):
                    image_paths.append(os.path.join(class_dir, filename))
                    labels.append(label_idx)
    return image_paths, labels

# クラス定義
class_names = ['dog', 'cat']  # クラス名のリスト
num_classes = len(class_names)

# ラベルマッピング
id2label = {i: name for i, name in enumerate(class_names)}
label2id = {name: i for i, name in enumerate(class_names)}

print(f"クラス数: {num_classes}")
print(f"id2label: {id2label}")
print(f"label2id: {label2id}")

In [ ]:
# ======================================
# モデルとプロセッサの準備
# ======================================
model_name = "google/vit-base-patch16-224"

# purocessorの準備
processor = ViTImageProcessor.from_pretrained(model_name)

# モデルの準備
# ignore_mismatched_sizes=Trueを指定して、事前学習済みモデルの重みをロードする際にサイズの不一致を無視する
# 事前学習済みモデルは1000クラス、今回は2クラスなので、出力層のサイズが異なるため
model = ViTForImageClassification.from_pretrained(
    model_name,
    num_labels=num_classes,
    id2label=id2label,
    label2id=label2id,
    ignore_mismatched_sizes=True
)

print(f"モデルのクラス数: {model.config.num_labels}")
print(f"パラメータ数": {sum(p.numel() for p in model.parameters()):,}")

In [ ]:
# ======================================
# 訓練設定
# ======================================
training_args = TrainingArguments(
    # 出力ディレクトリ(モデル保存先)
    output_dir = "./vit_finetuned",

    # バッチサイズ
    # GPUメモリに応じて調整
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,

    # エポック数
    # 過学習を防ぐために少な目に設定
    num_train_epochs=5,

    # 評価と保存のタイミング
    evaluation_strategy="epoch",
    save_strategy="epoch",

    # 学習率の設定
    # 学習率は小さめに設定することで、事前学習済みモデルの重みを大きく変えずに微調整する
    learning_rate=2e-5,

    #　正則化
    weight_decay=0.01,

    # ログの設定
    logging_dir="./logs",
    logging_steps=10,

    # ベストモデルの保存
    load_best_model_at_end=True,
    metric_for_best_model="accuracy",

    # 保存するチェックポイントの数の制限
    save_total_limit=2,

    # Trainerの内部設定
    remove_unused_columns=False,
)
print("訓練設定の準備が完了しました。")

In [ ]:
# ======================================
# 評価指標の定義
# ======================================

def compute_metrics(eval_pred):
    """
    評価指標の計算
    
    Args:
        eval_pred: 評価時の予測結果とラベルのタプル (predictions, labels)
        logits: モデルの出力 (未正規化のスコア) (N, num_classes)
        labels: 正解ラベル (N,)

    Returns:
        dict: 計算された評価指標の辞書
    """

    logits, labels = eval_pred
    # 予測ラベルを取得
    predictions = logits.argmax(axis=-1)

    # 精度の計算
    accuracy = accuracy_score(labels, predictions)

    # 精度、再現率、F1スコアの計算
    precision, recall, f1, _ = precision_recall_fscore_support(labels, predictions, average='weighted')

    return {
        'accuracy': accuracy,
        'precision': precision,
        'recall': recall,
        'f1': f1
    }

print("評価指標の定義が完了しました。")

In [ ]:
# ======================================
# Trainerの準備
# ======================================

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=eval_dataset,
    compute_metrics=compute_metrics,
)

# 訓練開始
print("訓練を開始します。")
trainer.train()

# 最終評価
print("最終評価を行います。")
results = trainer.evaluate()
for key, value in results.items():
    print(f"{key}: {value:.4f}")

## モデルの保存
trainer.save_model("./vit_finetuned_final")
processor.save_pretrained("./vit_finetuned_final")
print("モデルとプロセッサを保存しました。")

In [ ]:
# ======================================
# 戦略別の実装例
# ======================================

# ---
# 戦略1 分類ヘッドのみ学習
# ---

def freeze_encoder(model):
    """
    エンコーダ部分のパラメータを凍結する関数
    
    Args:
        model: ViTForImageClassificationモデル
    少量データ(1000枚未満)でのファインチューニングでは、事前学習済みモデルのエンコーダ部分を凍結し、分類ヘッドのみを学習することで過学習を防ぐことができます。
    """
    for name, param in model.named_parameters():
        if "classifier" not in name:
            param.requires_grad = False

    # 学習可能なパラメータの数を表示
    trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
    total = sum(p.numel() for p in model.parameters())
    print(f"学習可能なパラメータ数: {trainable:,} / {total:,} ({trainable/total:.2%})")

# 使用例
freeze_encoder(model)

In [ ]:
# ---
# 戦略2 全層学習
# ---

def freeze_lower_layers(model, num_layers_to_freeze=8):
    '''
    下位の層を凍結し、上位の層と分類ヘッドのみ学習可能にする

    Args:
        model: ViTモデル
        num_layers_to_freeze: 凍結する層数(0~11)
    
    中量データ(1000~5000枚)で有効
    '''

    for name,param in model.named_parameters():
        # 分類ヘッドは常に学習可能
        if "cllassifier" in name:
            param.requires_grad = True
            continue

        # 凍結する層を判定
        # ViTの層名は"encoder.layer.0", "encoder.layer.1", ...の形式
        should_freeze = False
        for i in range(num_layers_to_freeze):
            if f"encoder.layer.{i}." in name:
                should_freeze = True
                break
        param.requires_grad = not should_freeze

    # 学習可能なパラメータ数を確認
    trainable = sum(p.numel() for p in model.parameters() if p.requirees_grad)
    total = sum(p.numel() for p in model.parameters())
    print(f"凍結層数: {num_layers_to_freeze}")
    print(f"学習可能パラメータ: {trainable:,} / {total:,} ({trainable/total*100:.2f}%)")

    # 使用例: 下位8層を凍結、上位4層と分類ヘッドを学習
    freeze_lower_layers(model, num_layers_to_freeze=8)

In [ ]:
# ---
# 戦略3: 段階的学習(Gradual Unfreezing)
# ---

def gradual_unfreeze_training(model, train_dataset, val_dataset, processor):
    '''
    段階的に層を解凍しながら訓練

    Phase 1:分類ヘッドのみ(2 epocks)
    Phase 2:上位4層 + 分類ヘッド(2 epocks)
    Phase 3:全層
    '''